# 02 — Kalshi BTC Data Collection and Cleaning

This notebook collects, cleans, and stores Bitcoin-related Kalshi market data for the thesis sample.

The output will be a cleaned Kalshi dataset used later for comparison with Polymarket and Deribit.

In [49]:
# ============================================================
# Imports and project configuration
# ============================================================

import os
import re
import json
import time
import requests

import numpy as np
import pandas as pd

from importlib import reload
import config
reload(config)

from config import *

print("DOWNLOAD_KALSHI:", DOWNLOAD_KALSHI)
print("SAMPLE_START:", SAMPLE_START)
print("SAMPLE_END:", SAMPLE_END)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("FINAL_DIR:", FINAL_DIR)

for folder in [RAW_DIR, PROCESSED_DIR, FINAL_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DOWNLOAD_KALSHI: False
SAMPLE_START: 2024-01-01
SAMPLE_END: 2026-06-04
RAW_DIR: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw
PROCESSED_DIR: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed
FINAL_DIR: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final


In [51]:
# ============================================================
# Kalshi API settings
# ============================================================

KALSHI_BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
KALSHI_SERIES = "KXBTC"

KALSHI_HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
}

REQUEST_LIMIT = 1000
REQUEST_DELAY = 0.20

In [53]:
# ============================================================
# Kalshi API connection test
# ============================================================

test_response = requests.get(
    f"{KALSHI_BASE_URL}/exchange/status",
    headers=KALSHI_HEADERS,
    timeout=30,
)

print("Exchange status HTTP:", test_response.status_code)
print(test_response.text[:500])

series_response = requests.get(
    f"{KALSHI_BASE_URL}/series/{KALSHI_SERIES}",
    headers=KALSHI_HEADERS,
    timeout=30,
)

print("\nSeries HTTP:", series_response.status_code)
print(series_response.text[:500])

markets_response = requests.get(
    f"{KALSHI_BASE_URL}/markets",
    params={
        "series_ticker": KALSHI_SERIES,
        "status": "settled",
        "limit": 5,
    },
    headers=KALSHI_HEADERS,
    timeout=30,
)

print("\nMarkets HTTP:", markets_response.status_code)
print(markets_response.text[:500])

Exchange status HTTP: 200
{"exchange_active":true,"exchange_index_statuses":[{"description":"Default","exchange_active":true,"exchange_index":0,"intra_exchange_transfers_active":true,"trading_active":true},{"description":"Combos","exchange_active":true,"exchange_index":1,"intra_exchange_transfers_active":true,"trading_active":true}],"intra_exchange_transfers_active":true,"trading_active":true}

Series HTTP: 200
{"series":{"additional_prohibitions":["Persons who are employed by any of the Source Agencies are not permitted to trade on the Contract.","Persons who hold any material, non-public information on the Underlying are not permitted to trade on the Contract."],"category":"Crypto","contract_terms_url":"https://assets.kalshi.com/contract_terms/BTC.pdf","contract_url":"https://assets.kalshi.com/regulatory/product-certifications/BTC.pdf","exchange_index":0,"fee_multiplier":1,"fee_type":"quadratic","fre

Markets HTTP: 200
{"cursor":"CgwIz5yL1AYQoMjA7gESFktYQlRDLTI2QVVHMTgxNi1CNzIwNTA","

In [55]:
# ============================================================
# Kalshi sample window
# ============================================================

sample_start_ts = int(pd.Timestamp(SAMPLE_START, tz="UTC").timestamp())
sample_end_ts = int((pd.Timestamp(SAMPLE_END, tz="UTC") + pd.Timedelta(days=1)).timestamp())

sample_start_dt = pd.Timestamp(SAMPLE_START, tz="UTC")
sample_end_dt = pd.Timestamp(SAMPLE_END, tz="UTC") + pd.Timedelta(days=1)

print("Kalshi sample window UTC:")
print("  sample_start_ts:", sample_start_ts, sample_start_dt)
print("  sample_end_ts:", sample_end_ts, sample_end_dt)

Kalshi sample window UTC:
  sample_start_ts: 1704067200 2024-01-01 00:00:00+00:00
  sample_end_ts: 1780617600 2026-06-05 00:00:00+00:00


In [57]:
# ============================================================
# Download Kalshi KXBTC events
# ============================================================

kalshi_events_path = RAW_DIR / "kalshi_kxbtc_events.csv"


def fetch_kalshi_kxbtc_events(
    limit=200,
    max_pages=1000,
    request_delay=REQUEST_DELAY
):
    all_events = []
    cursor = None

    for page in range(1, max_pages + 1):
        params = {
            "series_ticker": KALSHI_SERIES,
            "status": "settled",
            "limit": limit,
        }

        if cursor:
            params["cursor"] = cursor

        response = requests.get(
            f"{KALSHI_BASE_URL}/events",
            params=params,
            headers=KALSHI_HEADERS,
            timeout=30,
        )

        if response.status_code != 200:
            raise RuntimeError(
                f"Kalshi events request failed on page {page}. "
                f"HTTP {response.status_code}: {response.text[:1000]}"
            )

        data = response.json()
        batch = data.get("events", [])

        if not batch:
            break

        all_events.extend(batch)
        cursor = data.get("cursor")

        if page == 1 or page % 10 == 0:
            print(
                f"Page {page:>4}: batch={len(batch):>4}, "
                f"total={len(all_events):>7}, cursor={bool(cursor)}"
            )

        if not cursor:
            break

        time.sleep(request_delay)

    return pd.DataFrame(all_events)


if DOWNLOAD_KALSHI:
    print("Downloading Kalshi KXBTC events...")

    df_kalshi_events_all = fetch_kalshi_kxbtc_events()

    if df_kalshi_events_all.empty:
        raise ValueError("No Kalshi KXBTC events were downloaded.")

    df_kalshi_events_all.to_csv(kalshi_events_path, index=False)
    print(f"Saved Kalshi KXBTC events to: {kalshi_events_path}")

else:
    if not kalshi_events_path.exists():
        raise FileNotFoundError(
            f"Kalshi events file not found: {kalshi_events_path}\n"
            "Set DOWNLOAD_KALSHI = True in config.py to download it."
        )

    df_kalshi_events_all = pd.read_csv(kalshi_events_path)
    print(f"Loaded Kalshi KXBTC events from: {kalshi_events_path}")

print("=" * 70)
print(f"Kalshi KXBTC events loaded: {len(df_kalshi_events_all):,}")
print("Columns:")
print(df_kalshi_events_all.columns.tolist())

Loaded Kalshi KXBTC events from: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw/kalshi_kxbtc_events.csv
Kalshi KXBTC events loaded: 9,695
Columns:
['available_on_brokers', 'category', 'collateral_return_type', 'event_ticker', 'last_updated_ts', 'mutually_exclusive', 'series_ticker', 'strike_date', 'strike_period', 'sub_title', 'title']


In [59]:
# ============================================================
# Filter daily midnight-ET KXBTC events
# ============================================================

df_kalshi_events_all["strike_date_dt"] = pd.to_datetime(
    df_kalshi_events_all["strike_date"],
    errors="coerce",
    utc=True
)

sample_start_dt = pd.Timestamp(SAMPLE_START, tz="UTC")
sample_end_dt = pd.Timestamp(SAMPLE_END, tz="UTC") + pd.Timedelta(days=1)

daily_event_mask = df_kalshi_events_all["event_ticker"].str.contains(
    r"^KXBTC-\d{2}[A-Z]{3}\d{2}00$",
    regex=True,
    na=False
)

sample_date_mask = (
    (df_kalshi_events_all["strike_date_dt"] >= sample_start_dt) &
    (df_kalshi_events_all["strike_date_dt"] < sample_end_dt)
)

df_kalshi_daily_events = (
    df_kalshi_events_all.loc[daily_event_mask & sample_date_mask]
    .sort_values("strike_date_dt")
    .reset_index(drop=True)
)

kalshi_daily_events_path = RAW_DIR / "kalshi_kxbtc_daily_events.csv"
df_kalshi_daily_events.to_csv(kalshi_daily_events_path, index=False)

print("=" * 70)
print("All KXBTC events:", len(df_kalshi_events_all))
print("Daily KXBTC events in sample:", len(df_kalshi_daily_events))

print("\nDaily event date range:")
print(df_kalshi_daily_events["strike_date_dt"].min())
print(df_kalshi_daily_events["strike_date_dt"].max())

print("\nSample subtitles:")
print(df_kalshi_daily_events["sub_title"].value_counts().head(10))

display(
    df_kalshi_daily_events[
        ["event_ticker", "strike_date", "strike_date_dt", "sub_title", "title"]
    ].head(20)
)

display(
    df_kalshi_daily_events[
        ["event_ticker", "strike_date", "strike_date_dt", "sub_title", "title"]
    ].tail(20)
)

All KXBTC events: 9695
Daily KXBTC events in sample: 486

Daily event date range:
2025-02-04 05:00:00+00:00
2026-06-04 04:00:00+00:00

Sample subtitles:
sub_title
On Feb 4, 2025 at 12am EST     1
On Dec 5, 2025 at 12am EST     1
On Jan 2, 2026 at 12am EST     1
On Jan 1, 2026 at 12am EST     1
On Dec 31, 2025 at 12am EST    1
On Dec 30, 2025 at 12am EST    1
On Dec 29, 2025 at 12am EST    1
On Dec 28, 2025 at 12am EST    1
On Dec 27, 2025 at 12am EST    1
On Dec 26, 2025 at 12am EST    1
Name: count, dtype: int64


,event_ticker,strike_date,strike_date_dt,sub_title,title
0,KXBTC-25FEB0400,2025-02-04T05:00:00Z,2025-02-04 05:00:00+00:00,"On Feb 4, 2025 at 12am EST","Bitcoin price range on Feb 4, 2025 at 12am EST?"
1,KXBTC-25FEB0500,2025-02-05T05:00:00Z,2025-02-05 05:00:00+00:00,"On Feb 5, 2025 at 12am EST","Bitcoin price range on Feb 5, 2025 at 12am EST?"
2,KXBTC-25FEB0600,2025-02-06T05:00:00Z,2025-02-06 05:00:00+00:00,"On Feb 6, 2025 at 12am EST","Bitcoin price range on Feb 6, 2025 at 12am EST?"
3,KXBTC-25FEB0700,2025-02-07T05:00:00Z,2025-02-07 05:00:00+00:00,"On Feb 7, 2025 at 12am EST","Bitcoin price range on Feb 7, 2025 at 12am EST?"
4,KXBTC-25FEB0800,2025-02-08T05:00:00Z,2025-02-08 05:00:00+00:00,"On Feb 8, 2025 at 12am EST","Bitcoin price range on Feb 8, 2025 at 12am EST?"
5,KXBTC-25FEB0900,2025-02-09T05:00:00Z,2025-02-09 05:00:00+00:00,"On Feb 9, 2025 at 12am EST","Bitcoin price range on Feb 9, 2025 at 12am EST?"
6,KXBTC-25FEB1000,2025-02-10T05:00:00Z,2025-02-10 05:00:00+00:00,"On Feb 10, 2025 at 12am EST","Bitcoin price range on Feb 10, 2025 at 12am EST?"
7,KXBTC-25FEB1100,2025-02-11T05:00:00Z,2025-02-11 05:00:00+00:00,"On Feb 11, 2025 at 12am EST","Bitcoin price range on Feb 11, 2025 at 12am EST?"
8,KXBTC-25FEB1200,2025-02-12T05:00:00Z,2025-02-12 05:00:00+00:00,"On Feb 12, 2025 at 12am EST","Bitcoin price range on Feb 12, 2025 at 12am EST?"
9,KXBTC-25FEB1300,2025-02-13T05:00:00Z,2025-02-13 05:00:00+00:00,"On Feb 13, 2025 at 12am EST","Bitcoin price range on Feb 13, 2025 at 12am EST?"


,event_ticker,strike_date,strike_date_dt,sub_title,title
466,KXBTC-26MAY1600,2026-05-16T04:00:00Z,2026-05-16 04:00:00+00:00,"On May 16, 2026 at 12am EDT","Bitcoin price range on May 16, 2026 at 12am EDT?"
467,KXBTC-26MAY1700,2026-05-17T04:00:00Z,2026-05-17 04:00:00+00:00,"On May 17, 2026 at 12am EDT","Bitcoin price range on May 17, 2026 at 12am EDT?"
468,KXBTC-26MAY1800,2026-05-18T04:00:00Z,2026-05-18 04:00:00+00:00,"On May 18, 2026 at 12am EDT","Bitcoin price range on May 18, 2026 at 12am EDT?"
469,KXBTC-26MAY1900,2026-05-19T04:00:00Z,2026-05-19 04:00:00+00:00,"On May 19, 2026 at 12am EDT","Bitcoin price range on May 19, 2026 at 12am EDT?"
470,KXBTC-26MAY2000,2026-05-20T04:00:00Z,2026-05-20 04:00:00+00:00,"On May 20, 2026 at 12am EDT","Bitcoin price range on May 20, 2026 at 12am EDT?"
471,KXBTC-26MAY2100,2026-05-21T04:00:00Z,2026-05-21 04:00:00+00:00,"On May 21, 2026 at 12am EDT","BTC price range on May 21, 2026 at 12am EDT?"
472,KXBTC-26MAY2200,2026-05-22T04:00:00Z,2026-05-22 04:00:00+00:00,"On May 22, 2026 at 12am EDT","BTC price range on May 22, 2026 at 12am EDT?"
473,KXBTC-26MAY2300,2026-05-23T04:00:00Z,2026-05-23 04:00:00+00:00,"On May 23, 2026 at 12am EDT","BTC price range on May 23, 2026 at 12am EDT?"
474,KXBTC-26MAY2400,2026-05-24T04:00:00Z,2026-05-24 04:00:00+00:00,"On May 24, 2026 at 12am EDT","BTC price range on May 24, 2026 at 12am EDT?"
475,KXBTC-26MAY2500,2026-05-25T04:00:00Z,2026-05-25 04:00:00+00:00,"On May 25, 2026 at 12am EDT","BTC price range on May 25, 2026 at 12am EDT?"


In [63]:
# ============================================================
# Download daily KXBTC markets from historical endpoint
# ============================================================

kalshi_daily_raw_path = RAW_DIR / "kalshi_kxbtc_daily_raw_markets.csv"


def fetch_historical_markets_for_event(event_ticker, limit=1000):
    response = requests.get(
        f"{KALSHI_BASE_URL}/historical/markets",
        params={
            "event_ticker": event_ticker,
            "limit": limit,
        },
        headers=KALSHI_HEADERS,
        timeout=30,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Kalshi historical markets request failed for {event_ticker}. "
            f"HTTP {response.status_code}: {response.text[:1000]}"
        )

    return response.json().get("markets", [])


if DOWNLOAD_KALSHI:
    daily_market_rows = []
    event_tickers = df_kalshi_daily_events["event_ticker"].tolist()

    print(f"Downloading historical markets for {len(event_tickers):,} daily KXBTC events...")

    for i, event_ticker in enumerate(event_tickers, start=1):
        markets = fetch_historical_markets_for_event(event_ticker)
        daily_market_rows.extend(markets)

        if i == 1 or i % 25 == 0 or i == len(event_tickers):
            print(
                f"Processed {i:>4}/{len(event_tickers):>4} events | "
                f"last_event={event_ticker} | "
                f"last_batch={len(markets):>4} | "
                f"total markets={len(daily_market_rows):,}"
            )

        time.sleep(REQUEST_DELAY)

    df_kalshi_raw = pd.DataFrame(daily_market_rows)

    if df_kalshi_raw.empty:
        raise ValueError("No Kalshi daily KXBTC historical markets were downloaded.")

    df_kalshi_raw.to_csv(kalshi_daily_raw_path, index=False)
    print(f"\nSaved daily Kalshi raw markets to: {kalshi_daily_raw_path}")

else:
    if not kalshi_daily_raw_path.exists():
        raise FileNotFoundError(
            f"Daily Kalshi raw file not found: {kalshi_daily_raw_path}\n"
            "Set DOWNLOAD_KALSHI = True in config.py to download it."
        )

    df_kalshi_raw = pd.read_csv(kalshi_daily_raw_path)
    print(f"Loaded daily Kalshi raw markets from: {kalshi_daily_raw_path}")

df_kalshi_raw["close_time_dt"] = pd.to_datetime(
    df_kalshi_raw["close_time"],
    errors="coerce",
    utc=True
)

print("=" * 70)
print(f"Raw Kalshi daily markets: {len(df_kalshi_raw):,}")
print(f"Unique tickers: {df_kalshi_raw['ticker'].nunique():,}")
print(f"Unique events: {df_kalshi_raw['event_ticker'].nunique():,}")

print("\nClose time range:")
print(df_kalshi_raw["close_time_dt"].min())
print(df_kalshi_raw["close_time_dt"].max())

print("\nStrike types:")
print(df_kalshi_raw["strike_type"].value_counts(dropna=False))

print("\nMarkets per event summary:")
display(
    df_kalshi_raw.groupby("event_ticker")["ticker"]
    .nunique()
    .describe()
)

display(df_kalshi_raw.head(10))

Loaded daily Kalshi raw markets from: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw/kalshi_kxbtc_daily_raw_markets.csv
Raw Kalshi daily markets: 34,172
Unique tickers: 34,172
Unique events: 427

Close time range:
2025-02-04 05:00:00+00:00
2026-04-06 04:00:00+00:00

Strike types:
strike_type
between    33311
greater      427
less         427
NaN            7
Name: count, dtype: int64

Markets per event summary:


count    427.000000
mean      80.028103
std       23.327414
min       75.000000
25%       75.000000
50%       75.000000
75%       75.000000
max      188.000000
Name: ticker, dtype: float64

,can_close_early,close_time,created_time,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,fractional_trading_enabled,last_price_dollars,...,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title,cap_strike,close_time_dt
0,True,2025-02-04T05:00:00Z,2025-02-02T10:06:07.363667Z,KXBTC-25FEB0400,2025-02-04T05:05:00Z,2025-02-11T05:00:00Z,NaN,108749.99,True,0.0,...,2026-02-19T08:48:16.628714Z,0.0,0.0,0.02,0.0,0.0,0.0,"$108,750 or above",NaN,2025-02-04 05:00:00+00:00
1,True,2025-02-04T05:00:00Z,2025-02-02T10:06:07.363667Z,KXBTC-25FEB0400,2025-02-04T05:05:00Z,2025-02-11T05:00:00Z,NaN,108500.00,True,0.0,...,2026-02-19T08:48:16.628714Z,0.0,0.0,1.00,0.0,0.0,0.0,"$108,500 to 108,749.99",108749.99,2025-02-04 05:00:00+00:00
2,True,2025-02-04T05:00:00Z,2025-02-02T10:06:07.363667Z,KXBTC-25FEB0400,2025-02-04T05:05:00Z,2025-02-11T05:00:00Z,NaN,108250.00,True,0.0,...,2026-02-19T08:48:16.628714Z,0.0,0.0,1.00,0.0,0.0,0.0,"$108,250 to 108,499.99",108499.99,2025-02-04 05:00:00+00:00
3,True,2025-02-04T05:00:00Z,2025-02-02T10:06:07.363667Z,KXBTC-25FEB0400,2025-02-04T05:05:00Z,2025-02-11T05:00:00Z,NaN,108000.00,True,0.0,...,2026-02-19T08:48:16.628714Z,0.0,0.0,0.02,0.0,0.0,0.0,"$108,000 to 108,249.99",108249.99,2025-02-04 05:00:00+00:00
4,True,2025-02-04T05:00:00Z,2025-02-02T10:06:07.363667Z,KXBTC-25FEB0400,2025-02-04T05:05:00Z,2025-02-11T05:00:00Z,NaN,107750.00,True,0.0,...,2026-02-19T08:48:16.628714Z,0.0,0.0,0.02,0.0,0.0,0.0,"$107,750 to 107,999.99",107999.99,2025-02-04 05:00:00+00:00
5,True,2025-02-04T05:00:00Z,2025-02-02T10:06:07.363667Z,KXBTC-25FEB0400,2025-02-04T05:05:00Z,2025-02-11T05:00:00Z,NaN,107500.00,True,0.0,...,2026-02-19T08:48:16.628714Z,0.0,0.0,0.02,0.0,0.0,0.0,"$107,500 to 107,749.99",107749.99,2025-02-04 05:00:00+00:00
6,True,2025-02-04T05:00:00Z,2025-02-02T10:06:07.363667Z,KXBTC-25FEB0400,2025-02-04T05:05:00Z,2025-02-11T05:00:00Z,NaN,107250.00,True,0.0,...,2026-02-19T08:48:16.628714Z,0.0,0.0,0.02,0.0,0.0,0.0,"$107,250 to 107,499.99",107499.99,2025-02-04 05:00:00+00:00
7,True,2025-02-04T05:00:00Z,2025-02-02T10:06:07.363667Z,KXBTC-25FEB0400,2025-02-04T05:05:00Z,2025-02-11T05:00:00Z,NaN,107000.00,True,0.0,...,2026-02-19T08:48:16.628714Z,0.0,0.0,0.02,0.0,0.0,0.0,"$107,000 to 107,249.99",107249.99,2025-02-04 05:00:00+00:00
8,True,2025-02-04T05:00:00Z,2025-02-02T10:06:07.363667Z,KXBTC-25FEB0400,2025-02-04T05:05:00Z,2025-02-11T05:00:00Z,NaN,106750.00,True,0.0,...,2026-02-19T08:48:16.628714Z,0.0,0.0,0.02,0.0,0.0,0.0,"$106,750 to 106,999.99",106999.99,2025-02-04 05:00:00+00:00
9,True,2025-02-04T05:00:00Z,2025-02-02T10:06:07.363666Z,KXBTC-25FEB0400,2025-02-04T05:05:00Z,2025-02-11T05:00:00Z,NaN,106500.00,True,0.0,...,2026-02-19T08:48:16.628714Z,0.0,0.0,0.02,0.0,0.0,0.0,"$106,500 to 106,749.99",106749.99,2025-02-04 05:00:00+00:00


In [95]:
# ============================================================
# Complete Kalshi daily raw dataset: historical + recent live
# ============================================================

kalshi_daily_raw_complete_path = (
    RAW_DIR / "kalshi_kxbtc_daily_raw_markets_complete.csv"
)


def fetch_live_markets_for_event(event_ticker, limit=1000):
    response = requests.get(
        f"{KALSHI_BASE_URL}/markets",
        params={
            "event_ticker": event_ticker,
            "limit": limit,
        },
        headers=KALSHI_HEADERS,
        timeout=30,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Kalshi live markets request failed for {event_ticker}. "
            f"HTTP {response.status_code}: {response.text[:1000]}"
        )

    return response.json().get("markets", [])


all_daily_events = set(
    df_kalshi_daily_events["event_ticker"]
    .dropna()
    .unique()
)


if DOWNLOAD_KALSHI:
    # --------------------------------------------------------
    # Combine historical markets with recent live markets
    # --------------------------------------------------------

    historical_events = set(
        df_kalshi_raw["event_ticker"]
        .dropna()
        .unique()
    )

    missing_live_events = sorted(
        all_daily_events - historical_events
    )

    print("Daily events expected:", len(all_daily_events))
    print(
        "Events from historical endpoint:",
        len(historical_events),
    )
    print(
        "Events missing from historical endpoint:",
        len(missing_live_events),
    )

    if missing_live_events:
        print("\nMissing event range:")
        print(missing_live_events[:5])
        print(missing_live_events[-5:])

    live_market_rows = []

    print(
        f"\nDownloading live markets for "
        f"{len(missing_live_events):,} missing events..."
    )

    for i, event_ticker in enumerate(
        missing_live_events,
        start=1,
    ):
        markets = fetch_live_markets_for_event(
            event_ticker
        )

        live_market_rows.extend(markets)

        if (
            i == 1
            or i % 10 == 0
            or i == len(missing_live_events)
        ):
            print(
                f"Processed "
                f"{i:>3}/{len(missing_live_events):>3} "
                f"missing events | "
                f"last_event={event_ticker} | "
                f"last_batch={len(markets):>4} | "
                f"live markets={len(live_market_rows):,}"
            )

        time.sleep(REQUEST_DELAY)

    df_kalshi_live_missing = pd.DataFrame(
        live_market_rows
    )

    df_kalshi_raw_complete = (
        pd.concat(
            [
                df_kalshi_raw,
                df_kalshi_live_missing,
            ],
            ignore_index=True,
        )
        .drop_duplicates(
            subset=["ticker"],
            keep="last",
        )
        .copy()
    )

    complete_events = set(
        df_kalshi_raw_complete["event_ticker"]
        .dropna()
        .unique()
    )

    events_still_missing = sorted(
        all_daily_events - complete_events
    )

    if events_still_missing:
        raise ValueError(
            f"{len(events_still_missing):,} daily events "
            "are still missing after combining historical "
            f"and live markets. Examples: "
            f"{events_still_missing[:10]}"
        )

    if df_kalshi_raw_complete["ticker"].duplicated().any():
        raise ValueError(
            "Duplicate tickers found in complete "
            "Kalshi raw dataset."
        )

    # Save only after all completeness checks pass.
    df_kalshi_raw_complete.to_csv(
        kalshi_daily_raw_complete_path,
        index=False,
    )

    print(
        "\nSaved complete Kalshi raw markets to:",
        kalshi_daily_raw_complete_path,
    )

else:
    # --------------------------------------------------------
    # Load the previously validated complete local file
    # --------------------------------------------------------

    if not kalshi_daily_raw_complete_path.exists():
        raise FileNotFoundError(
            "Complete Kalshi raw file not found:\n"
            f"{kalshi_daily_raw_complete_path}\n"
            "Restore the complete file or set "
            "DOWNLOAD_KALSHI = True."
        )

    df_kalshi_raw_complete = pd.read_csv(
        kalshi_daily_raw_complete_path,
        low_memory=False,
    )

    print(
        "Loaded complete Kalshi raw markets from:",
        kalshi_daily_raw_complete_path,
    )


# ------------------------------------------------------------
# Common validation and diagnostics
# ------------------------------------------------------------

df_kalshi_raw_complete = (
    df_kalshi_raw_complete
    .drop_duplicates(
        subset=["ticker"],
        keep="last",
    )
    .copy()
)

df_kalshi_raw_complete["close_time_dt"] = (
    pd.to_datetime(
        df_kalshi_raw_complete["close_time"],
        errors="coerce",
        utc=True,
    )
)

complete_events = set(
    df_kalshi_raw_complete["event_ticker"]
    .dropna()
    .unique()
)

events_missing_from_complete = sorted(
    all_daily_events - complete_events
)

duplicate_tickers = (
    df_kalshi_raw_complete["ticker"]
    .duplicated()
    .sum()
)

print("=" * 70)
print("Complete Kalshi daily raw dataset")
print("=" * 70)

print(
    f"Complete raw markets: "
    f"{len(df_kalshi_raw_complete):,}"
)
print(
    f"Unique tickers: "
    f"{df_kalshi_raw_complete['ticker'].nunique():,}"
)
print(
    f"Unique events: "
    f"{df_kalshi_raw_complete['event_ticker'].nunique():,}"
)
print(
    f"Missing expected events: "
    f"{len(events_missing_from_complete):,}"
)
print(f"Duplicate tickers: {duplicate_tickers:,}")

print("\nClose-time range:")
print(df_kalshi_raw_complete["close_time_dt"].min())
print(df_kalshi_raw_complete["close_time_dt"].max())

print("\nStrike types:")
print(
    df_kalshi_raw_complete[
        "strike_type"
    ].value_counts(dropna=False)
)

print("\nMarkets per event summary:")
display(
    df_kalshi_raw_complete
    .groupby("event_ticker")["ticker"]
    .nunique()
    .describe()
)

# Fixed-sample validation.
assert len(all_daily_events) == 486
assert len(df_kalshi_raw_complete) == 45_264
assert df_kalshi_raw_complete["ticker"].nunique() == 45_264
assert df_kalshi_raw_complete["event_ticker"].nunique() == 486
assert duplicate_tickers == 0
assert not events_missing_from_complete

display(df_kalshi_raw_complete.head(10))

print("\nComplete Kalshi raw dataset validation passed.")

Loaded complete Kalshi raw markets from: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw/kalshi_kxbtc_daily_raw_markets_complete.csv
Complete Kalshi daily raw dataset
Complete raw markets: 45,264
Unique tickers: 45,264
Unique events: 486
Missing expected events: 0
Duplicate tickers: 0

Close-time range:
2025-02-04 05:00:00+00:00
2026-06-04 04:00:00+00:00

Strike types:
strike_type
between    44285
greater      486
less         486
NaN            7
Name: count, dtype: int64

Markets per event summary:


count    486.000000
mean      93.135802
std       41.520906
min       75.000000
25%       75.000000
50%       75.000000
75%       75.000000
max      188.000000
Name: ticker, dtype: float64

,can_close_early,close_time,created_time,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,fractional_trading_enabled,last_price_dollars,...,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title,cap_strike,close_time_dt
0,True,2025-02-04 05:00:00+00:00,2025-02-02 10:06:07.363659+00:00,KXBTC-25FEB0400,2025-02-04 05:05:00+00:00,2025-02-11 05:00:00+00:00,NaN,90500.0,True,0.0,...,2026-02-19 08:48:16.628714+00:00,0.0,0.0,0.02,0.0,0.0,0.0,"$90,500 to 90,749.99",90749.99,2025-02-04 05:00:00+00:00
1,True,2025-02-04 05:00:00+00:00,2025-02-02 10:06:07.363659+00:00,KXBTC-25FEB0400,2025-02-04 05:05:00+00:00,2025-02-11 05:00:00+00:00,NaN,90750.0,True,0.0,...,2026-02-19 08:48:16.628714+00:00,0.0,0.0,0.02,0.0,0.0,0.0,"$90,750 to 90,999.99",90999.99,2025-02-04 05:00:00+00:00
2,True,2025-02-04 05:00:00+00:00,2025-02-02 10:06:07.363659+00:00,KXBTC-25FEB0400,2025-02-04 05:05:00+00:00,2025-02-11 05:00:00+00:00,NaN,91000.0,True,0.0,...,2026-02-19 08:48:16.628714+00:00,0.0,0.0,0.02,0.0,0.0,0.0,"$91,000 to 91,249.99",91249.99,2025-02-04 05:00:00+00:00
3,True,2025-02-04 05:00:00+00:00,2025-02-02 10:06:07.363659+00:00,KXBTC-25FEB0400,2025-02-04 05:05:00+00:00,2025-02-11 05:00:00+00:00,NaN,91250.0,True,0.0,...,2026-02-19 08:48:16.628714+00:00,0.0,0.0,0.02,0.0,0.0,0.0,"$91,250 to 91,499.99",91499.99,2025-02-04 05:00:00+00:00
4,True,2025-02-04 05:00:00+00:00,2025-02-02 10:06:07.363659+00:00,KXBTC-25FEB0400,2025-02-04 05:05:00+00:00,2025-02-11 05:00:00+00:00,NaN,91500.0,True,0.0,...,2026-02-19 08:48:16.628714+00:00,0.0,0.0,0.02,0.0,0.0,0.0,"$91,500 to 91,749.99",91749.99,2025-02-04 05:00:00+00:00
5,True,2025-02-04 05:00:00+00:00,2025-02-02 10:06:07.363659+00:00,KXBTC-25FEB0400,2025-02-04 05:05:00+00:00,2025-02-11 05:00:00+00:00,NaN,91750.0,True,0.0,...,2026-02-19 08:48:16.628714+00:00,0.0,0.0,0.02,0.0,0.0,0.0,"$91,750 to 91,999.99",91999.99,2025-02-04 05:00:00+00:00
6,True,2025-02-04 05:00:00+00:00,2025-02-02 10:06:07.363659+00:00,KXBTC-25FEB0400,2025-02-04 05:05:00+00:00,2025-02-11 05:00:00+00:00,NaN,92000.0,True,0.0,...,2026-02-19 08:48:16.628714+00:00,0.0,0.0,0.02,0.0,0.0,0.0,"$92,000 to 92,249.99",92249.99,2025-02-04 05:00:00+00:00
7,True,2025-02-04 05:00:00+00:00,2025-02-02 10:06:07.363660+00:00,KXBTC-25FEB0400,2025-02-04 05:05:00+00:00,2025-02-11 05:00:00+00:00,NaN,92250.0,True,0.0,...,2026-02-19 08:48:16.628714+00:00,0.0,0.0,0.02,0.0,0.0,0.0,"$92,250 to 92,499.99",92499.99,2025-02-04 05:00:00+00:00
8,True,2025-02-04 05:00:00+00:00,2025-02-02 10:06:07.363660+00:00,KXBTC-25FEB0400,2025-02-04 05:05:00+00:00,2025-02-11 05:00:00+00:00,NaN,92500.0,True,0.0,...,2026-02-19 08:48:16.628714+00:00,0.0,0.0,0.02,0.0,0.0,0.0,"$92,500 to 92,749.99",92749.99,2025-02-04 05:00:00+00:00
9,True,2025-02-04 05:00:00+00:00,2025-02-02 10:06:07.363660+00:00,KXBTC-25FEB0400,2025-02-04 05:05:00+00:00,2025-02-11 05:00:00+00:00,NaN,92750.0,True,0.0,...,2026-02-19 08:48:16.628714+00:00,0.0,0.0,0.02,0.0,0.0,0.0,"$92,750 to 92,999.99",92999.99,2025-02-04 05:00:00+00:00



Complete Kalshi raw dataset validation passed.


In [99]:
# ============================================================
# Clean Kalshi raw daily markets
# ============================================================

df_kalshi_meta = df_kalshi_raw_complete.copy()

date_cols = [
    "close_time",
    "created_time",
    "expected_expiration_time",
    "expiration_time",
    "latest_expiration_time",
    "updated_time",
]

for col in date_cols:
    if col in df_kalshi_meta.columns:
        df_kalshi_meta[col] = pd.to_datetime(
            df_kalshi_meta[col],
            errors="coerce",
            utc=True
        )

numeric_cols = [
    "floor_strike",
    "cap_strike",
    "expiration_value",
    "last_price_dollars",
    "previous_price_dollars",
    "yes_bid_dollars",
    "yes_ask_dollars",
    "no_bid_dollars",
    "no_ask_dollars",
    "volume_fp",
    "volume_24h_fp",
    "open_interest_fp",
    "liquidity_dollars",
]

for col in numeric_cols:
    if col in df_kalshi_meta.columns:
        df_kalshi_meta[col] = pd.to_numeric(
            df_kalshi_meta[col],
            errors="coerce"
        )

df_kalshi_meta["date"] = df_kalshi_meta["close_time"].dt.normalize()
df_kalshi_meta["market_source"] = "kalshi"
df_kalshi_meta["series_ticker"] = KALSHI_SERIES

df_kalshi_meta = (
    df_kalshi_meta
    .sort_values(["close_time", "event_ticker", "strike_type", "floor_strike", "cap_strike"])
    .reset_index(drop=True)
)

print("=" * 70)
print("Cleaned Kalshi metadata")
print("=" * 70)
print(f"Rows: {len(df_kalshi_meta):,}")
print(f"Unique tickers: {df_kalshi_meta['ticker'].nunique():,}")
print(f"Unique events: {df_kalshi_meta['event_ticker'].nunique():,}")

print("\nDate range:")
print(df_kalshi_meta["close_time"].min())
print(df_kalshi_meta["close_time"].max())

print("\nStrike types:")
print(df_kalshi_meta["strike_type"].value_counts(dropna=False))

print("\nMissing values:")
print(
    df_kalshi_meta[
        ["ticker", "event_ticker", "close_time", "strike_type", "floor_strike", "cap_strike"]
    ].isna().sum()
)

Cleaned Kalshi metadata
Rows: 45,264
Unique tickers: 45,264
Unique events: 486

Date range:
2025-02-04 05:00:00+00:00
2026-06-04 04:00:00+00:00

Strike types:
strike_type
between    44285
greater      486
less         486
NaN            7
Name: count, dtype: int64

Missing values:
ticker            0
event_ticker      0
close_time        0
strike_type       7
floor_strike    493
cap_strike      493
dtype: int64


In [101]:
# ============================================================
# Define Kalshi valid-strike metadata universe
# ============================================================

MAIN_STRIKE_MIN = 40_000
MAIN_STRIKE_MAX = 200_000

df_kalshi = df_kalshi_meta.copy()

df_kalshi["strike_reference"] = df_kalshi["floor_strike"].fillna(
    df_kalshi["cap_strike"]
)

df_kalshi["valid_strike"] = (
    df_kalshi["strike_type"].notna()
    & df_kalshi["strike_reference"].notna()
)

df_kalshi["main_sample"] = (
    df_kalshi["valid_strike"]
    & df_kalshi["strike_reference"].between(
        MAIN_STRIKE_MIN,
        MAIN_STRIKE_MAX,
        inclusive="both",
    )
)

df_kalshi_main = df_kalshi.loc[df_kalshi["main_sample"]].copy()

# Name expected by the timestamped-trade pipeline.
df_kalshi_main_all = df_kalshi_main.copy()

print("=" * 70)
print("Kalshi valid-strike metadata universe")
print("=" * 70)
print(f"Full metadata rows: {len(df_kalshi):,}")
print(f"Main metadata rows: {len(df_kalshi_main):,}")
print(f"Main events: {df_kalshi_main['event_ticker'].nunique():,}")
print(f"Main tickers: {df_kalshi_main['ticker'].nunique():,}")

Kalshi valid-strike metadata universe
Full metadata rows: 45,264
Main metadata rows: 45,257
Main events: 486
Main tickers: 45,257


In [103]:
# ============================================================
# Validate Kalshi metadata universe
# ============================================================

metadata_checks = {
    "duplicate_tickers": df_kalshi_main["ticker"].duplicated().sum(),
    "missing_tickers": df_kalshi_main["ticker"].isna().sum(),
    "missing_events": df_kalshi_main["event_ticker"].isna().sum(),
    "missing_open_time": df_kalshi_main["open_time"].isna().sum(),
    "missing_close_time": df_kalshi_main["close_time"].isna().sum(),
    "invalid_time_order": (
        df_kalshi_main["open_time"] >= df_kalshi_main["close_time"]
    ).sum(),
    "invalid_strikes": (~df_kalshi_main["valid_strike"]).sum(),
}

display(pd.Series(metadata_checks, name="count"))

critical_checks = [
    "duplicate_tickers",
    "missing_tickers",
    "missing_events",
    "missing_open_time",
    "missing_close_time",
    "invalid_time_order",
    "invalid_strikes",
]

assert all(metadata_checks[key] == 0 for key in critical_checks)
print("Critical Kalshi metadata checks passed.")

duplicate_tickers     0
missing_tickers       0
missing_events        0
missing_open_time     0
missing_close_time    0
invalid_time_order    0
invalid_strikes       0
Name: count, dtype: int64

Critical Kalshi metadata checks passed.


In [105]:
# ============================================================
# Save processed Kalshi metadata datasets
# ============================================================

kalshi_metadata_path = (
    PROCESSED_DIR / "kalshi_kxbtc_daily_markets_metadata.csv"
)
kalshi_main_metadata_path = (
    PROCESSED_DIR / "kalshi_kxbtc_daily_markets_main.csv"
)

df_kalshi.to_csv(kalshi_metadata_path, index=False)
df_kalshi_main.to_csv(kalshi_main_metadata_path, index=False)

print("Saved full metadata to:", kalshi_metadata_path)
print("Saved main metadata to:", kalshi_main_metadata_path)

Saved full metadata to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/kalshi_kxbtc_daily_markets_metadata.csv
Saved main metadata to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/kalshi_kxbtc_daily_markets_main.csv


In [35]:
# ============================================================
# Download exact trades for all traded Kalshi markets
# ============================================================

import time
import pandas as pd
import requests

KALSHI_TRADE_DELAY = 0.10
CHECKPOINT_EVERY = 25

kalshi_trades_raw_path = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "kalshi_kxbtc_timestamped_trades.csv"
)

kalshi_trades_status_path = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "kalshi_kxbtc_timestamped_trades_status.csv"
)

df_traded_markets = df_kalshi_main_all.copy()

df_traded_markets["volume_fp_numeric"] = pd.to_numeric(
    df_traded_markets["volume_fp"],
    errors="coerce",
).fillna(0)

df_traded_markets = (
    df_traded_markets[
        df_traded_markets["volume_fp_numeric"] > 0
    ]
    .sort_values(["event_ticker", "ticker"])
    .reset_index(drop=True)
)

# Resume only tickers previously completed successfully.
if kalshi_trades_status_path.exists():
    previous_status = pd.read_csv(
        kalshi_trades_status_path,
        low_memory=False,
    )

    completed_tickers = set(
        previous_status.loc[
            previous_status["download_status"].isin(
                ["ok", "empty"]
            ),
            "ticker",
        ]
    )
else:
    completed_tickers = set()

markets_to_download = df_traded_markets[
    ~df_traded_markets["ticker"].isin(completed_tickers)
].copy()

print("=" * 70)
print("Kalshi exact historical trades download")
print("=" * 70)
print("Eligible traded markets:", len(df_traded_markets))
print("Already completed:", len(completed_tickers))
print("Remaining:", len(markets_to_download))
print("Raw trades path:", kalshi_trades_raw_path)
print("Status path:", kalshi_trades_status_path)


def fetch_all_historical_trades(ticker):
    all_trades = []
    cursor = None
    page = 0

    while True:
        page += 1

        params = {
            "ticker": ticker,
            "limit": 1000,
        }

        if cursor:
            params["cursor"] = cursor

        response = None

        for attempt in range(1, 4):
            try:
                response = requests.get(
                    f"{KALSHI_BASE_URL}/historical/trades",
                    params=params,
                    headers=KALSHI_HEADERS,
                    timeout=30,
                )

                if response.status_code == 200:
                    break

                if (
                    response.status_code == 429
                    or response.status_code >= 500
                ):
                    time.sleep(attempt)
                    continue

                return (
                    all_trades,
                    False,
                    page,
                    f"HTTP {response.status_code}: "
                    f"{response.text[:500]}",
                )

            except requests.exceptions.RequestException as exc:
                if attempt == 3:
                    return all_trades, False, page, str(exc)

                time.sleep(attempt)

        if response is None or response.status_code != 200:
            return all_trades, False, page, "Request failed"

        data = response.json()
        batch = data.get("trades", [])

        for trade in batch:
            trade["requested_ticker"] = ticker

        all_trades.extend(batch)

        cursor = data.get("cursor")

        if not cursor:
            break

        time.sleep(KALSHI_TRADE_DELAY)

    return all_trades, True, page, None


trade_buffer = []
status_buffer = []

for position, market in enumerate(
    markets_to_download.itertuples(index=False),
    start=1,
):
    trades, success, pages, error = (
        fetch_all_historical_trades(market.ticker)
    )

    trade_buffer.extend(trades)

    status_buffer.append(
        {
            "ticker": market.ticker,
            "event_ticker": market.event_ticker,
            "download_status": (
                "ok"
                if success and trades
                else "empty"
                if success
                else "error"
            ),
            "trades_returned": len(trades),
            "pages": pages,
            "error": error,
        }
    )

    should_flush = (
        position % CHECKPOINT_EVERY == 0
        or position == len(markets_to_download)
    )

    if should_flush:
        if trade_buffer:
            pd.DataFrame(trade_buffer).to_csv(
                kalshi_trades_raw_path,
                mode="a",
                header=not kalshi_trades_raw_path.exists(),
                index=False,
            )

        if status_buffer:
            pd.DataFrame(status_buffer).to_csv(
                kalshi_trades_status_path,
                mode="a",
                header=not kalshi_trades_status_path.exists(),
                index=False,
            )

        trade_buffer = []
        status_buffer = []

        print(
            f"Processed {position:>4}/"
            f"{len(markets_to_download):>4} remaining markets"
        )

    time.sleep(KALSHI_TRADE_DELAY)

print("\n" + "=" * 70)
print("Kalshi trade download completed")
print("=" * 70)
print("Raw trades saved to:", kalshi_trades_raw_path)
print("Download status saved to:", kalshi_trades_status_path)

Kalshi exact historical trades download
Eligible traded markets: 3809
Already completed: 0
Remaining: 3809
Raw trades path: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw/kalshi_kxbtc_timestamped_trades.csv
Status path: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw/kalshi_kxbtc_timestamped_trades_status.csv
Processed   25/3809 remaining markets
Processed   50/3809 remaining markets
Processed   75/3809 remaining markets
Processed  100/3809 remaining markets
Processed  125/3809 remaining markets
Processed  150/3809 remaining markets
Processed  175/3809 remaining markets
Processed  200/3809 remaining markets
Processed  225/3809 remaining markets
Processed  250/3809 remaining markets
Processed  275/3809 remaining markets
Processed  300/3809 remaining markets
Processed  325/3809 remaining markets
Processed  350/3809 remaining markets
Processed  375/3809 remaining markets
Processed  400/3809 remaining markets
Processed  425/3809 remaining markets
Processe

In [107]:
# ============================================================
# Validate downloaded Kalshi historical trades
# ============================================================

df_kalshi_trades_raw = pd.read_csv(
    kalshi_trades_raw_path,
    low_memory=False,
)

df_kalshi_trade_status = pd.read_csv(
    kalshi_trades_status_path,
    low_memory=False,
)

df_kalshi_trades_raw["created_time"] = pd.to_datetime(
    df_kalshi_trades_raw["created_time"],
    utc=True,
    errors="coerce",
)

df_kalshi_trades_raw["yes_price"] = pd.to_numeric(
    df_kalshi_trades_raw["yes_price_dollars"],
    errors="coerce",
)

df_kalshi_trades_raw["trade_count"] = pd.to_numeric(
    df_kalshi_trades_raw["count_fp"],
    errors="coerce",
)

trade_metadata = df_traded_markets[
    [
        "ticker",
        "event_ticker",
        "open_time",
        "close_time",
        "volume_fp_numeric",
    ]
].copy()

df_kalshi_trades_check = df_kalshi_trades_raw.merge(
    trade_metadata,
    on="ticker",
    how="left",
    validate="many_to_one",
)

volume_reconciliation = (
    df_kalshi_trades_check
    .groupby("ticker", as_index=False)
    .agg(
        downloaded_trades=("trade_id", "nunique"),
        downloaded_volume=("trade_count", "sum"),
        metadata_volume=("volume_fp_numeric", "first"),
    )
)

volume_reconciliation["volume_difference"] = (
    volume_reconciliation["downloaded_volume"]
    - volume_reconciliation["metadata_volume"]
)

volume_reconciliation["volume_reconciled"] = np.isclose(
    volume_reconciliation["downloaded_volume"],
    volume_reconciliation["metadata_volume"],
    atol=0.01,
    rtol=1e-8,
)

print("=" * 70)
print("Kalshi historical trade validation")
print("=" * 70)

print("Status rows:", len(df_kalshi_trade_status))
print(
    "Successful ticker downloads:",
    (df_kalshi_trade_status["download_status"] == "ok").sum(),
)
print("Raw trade rows:", len(df_kalshi_trades_raw))
print(
    "Unique trade IDs:",
    df_kalshi_trades_raw["trade_id"].nunique(),
)
print(
    "Duplicate trade IDs:",
    df_kalshi_trades_raw.duplicated("trade_id").sum(),
)

print(
    "Trades before market open:",
    (
        df_kalshi_trades_check["created_time"]
        < df_kalshi_trades_check["open_time"]
    ).sum(),
)

print(
    "Trades after market close:",
    (
        df_kalshi_trades_check["created_time"]
        > df_kalshi_trades_check["close_time"]
    ).sum(),
)

print(
    "Invalid prices:",
    (
        ~df_kalshi_trades_check["yes_price"]
        .between(0, 1, inclusive="both")
    ).sum(),
)

print(
    "Invalid trade counts:",
    (
        df_kalshi_trades_check["trade_count"].isna()
        | (df_kalshi_trades_check["trade_count"] <= 0)
    ).sum(),
)

print(
    "Volume reconciliation failures:",
    (~volume_reconciliation["volume_reconciled"]).sum(),
)

print("\nMarkets with volume discrepancies:")
display(
    volume_reconciliation.loc[
        ~volume_reconciliation["volume_reconciled"]
    ].sort_values("volume_difference")
)

assert (
    df_kalshi_trade_status["download_status"] == "ok"
).all()

assert not df_kalshi_trades_raw["trade_id"].duplicated().any()

assert df_kalshi_trades_check["event_ticker"].notna().all()

assert (
    df_kalshi_trades_check["created_time"]
    >= df_kalshi_trades_check["open_time"]
).all()

assert (
    df_kalshi_trades_check["created_time"]
    <= df_kalshi_trades_check["close_time"]
).all()

assert df_kalshi_trades_check["yes_price"].between(
    0, 1,
    inclusive="both",
).all()

print("\nCritical validation checks passed.")

Kalshi historical trade validation
Status rows: 3809
Successful ticker downloads: 3809
Raw trade rows: 124852
Unique trade IDs: 124852
Duplicate trade IDs: 0
Trades before market open: 0
Trades after market close: 0
Invalid prices: 0
Invalid trade counts: 0
Volume reconciliation failures: 7

Markets with volume discrepancies:


,ticker,downloaded_trades,downloaded_volume,metadata_volume,volume_difference,volume_reconciled
1898,KXBTC-25SEP2700-B109875,42,3427.0,3927.0,-500.0,False
1879,KXBTC-25SEP2400-B111875,146,16183.0,16211.0,-28.0,False
297,KXBTC-25AUG2500-B113125,14,2967.0,2992.0,-25.0,False
1809,KXBTC-25SEP1000-B111625,70,9866.0,9888.0,-22.0,False
1762,KXBTC-25SEP0100-B107625,36,2535.0,2554.0,-19.0,False
826,KXBTC-25JUL2800-B119625,17,1301.0,1309.0,-8.0,False
1880,KXBTC-25SEP2400-B112125,82,13449.0,13453.0,-4.0,False



Critical validation checks passed.


In [109]:
# ============================================================
# Construct timestamped Kalshi first-trade samples
# ============================================================

# ------------------------------------------------------------
# Validate metadata source
# ------------------------------------------------------------

expected_metadata_rows = 45_257
expected_metadata_events = 486

print("Metadata source:")
print(f"  Rows: {len(df_kalshi_main_all):,}")
print(f"  Events: {df_kalshi_main_all['event_ticker'].nunique():,}")

if len(df_kalshi_main_all) != expected_metadata_rows:
    raise ValueError(
        "Unexpected Kalshi main metadata size. "
        f"Expected {expected_metadata_rows:,}, "
        f"found {len(df_kalshi_main_all):,}. "
        "Check that df_kalshi_meta is constructed from "
        "df_kalshi_raw_complete."
    )

if df_kalshi_main_all["event_ticker"].nunique() != expected_metadata_events:
    raise ValueError(
        "Unexpected number of Kalshi metadata events. "
        f"Expected {expected_metadata_events:,}, "
        f"found {df_kalshi_main_all['event_ticker'].nunique():,}."
    )

if not df_kalshi_main_all["ticker"].is_unique:
    raise ValueError("Kalshi metadata contains duplicate tickers.")


# ------------------------------------------------------------
# Select earliest exact trade by reconciled ticker
# ------------------------------------------------------------

reconciled_tickers = set(
    volume_reconciliation.loc[
        volume_reconciliation["volume_reconciled"],
        "ticker",
    ]
)

df_kalshi_first_trades = (
    df_kalshi_trades_raw.loc[
        df_kalshi_trades_raw["ticker"].isin(reconciled_tickers)
    ]
    .sort_values(["ticker", "created_time", "trade_id"])
    .drop_duplicates(subset=["ticker"], keep="first")
    .copy()
)

df_kalshi_first_trades = df_kalshi_first_trades.rename(
    columns={
        "created_time": "observation_time",
        "yes_price": "prob_kalshi",
        "trade_count": "first_trade_size",
    }
)


# ------------------------------------------------------------
# Check trade-metadata coverage before merging
# ------------------------------------------------------------

trade_tickers = set(df_kalshi_first_trades["ticker"])
metadata_tickers = set(df_kalshi_main_all["ticker"])

unmatched_tickers = sorted(trade_tickers - metadata_tickers)

print("\nTrade-metadata coverage:")
print(f"  First-trade tickers: {len(trade_tickers):,}")
print(f"  Tickers without metadata: {len(unmatched_tickers):,}")

if unmatched_tickers:
    raise ValueError(
        f"{len(unmatched_tickers):,} traded tickers have no metadata. "
        f"Examples: {unmatched_tickers[:10]}. "
        "Rerun the complete historical + live metadata cells."
    )


# ------------------------------------------------------------
# Merge exact trades with contract metadata
# ------------------------------------------------------------

metadata_columns = [
    "ticker",
    "event_ticker",
    "open_time",
    "close_time",
    "strike_type",
    "floor_strike",
    "cap_strike",
    "strike_reference",
    "result",
    "expiration_value",
    "volume_fp",
    "title",
    "subtitle",
    "yes_sub_title",
]

trade_columns = [
    "ticker",
    "trade_id",
    "observation_time",
    "prob_kalshi",
    "first_trade_size",
    "is_block_trade",
    "taker_side",
    "taker_book_side",
    "taker_outcome_side",
]

df_kalshi_timestamped = (
    df_kalshi_first_trades[trade_columns]
    .merge(
        df_kalshi_main_all[metadata_columns],
        on="ticker",
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# Construct timing and validity variables
# ------------------------------------------------------------

for col in ["observation_time", "open_time", "close_time"]:
    df_kalshi_timestamped[col] = pd.to_datetime(
        df_kalshi_timestamped[col],
        utc=True,
        errors="coerce",
    )

df_kalshi_timestamped["minutes_after_open"] = (
    (
        df_kalshi_timestamped["observation_time"]
        - df_kalshi_timestamped["open_time"]
    ).dt.total_seconds()
    / 60
)

df_kalshi_timestamped["minutes_to_close"] = (
    (
        df_kalshi_timestamped["close_time"]
        - df_kalshi_timestamped["observation_time"]
    ).dt.total_seconds()
    / 60
)

df_kalshi_timestamped["tau_raw"] = (
    df_kalshi_timestamped["minutes_to_close"]
    / (365 * 24 * 60)
)

df_kalshi_timestamped["date"] = (
    df_kalshi_timestamped["observation_time"].dt.normalize()
)

df_kalshi_timestamped["price_source"] = "first_timestamped_trade"

df_kalshi_timestamped["trade_within_15m"] = (
    df_kalshi_timestamped["minutes_after_open"].between(
        0,
        15,
        inclusive="both",
    )
)

df_kalshi_timestamped["trade_within_30m"] = (
    df_kalshi_timestamped["minutes_after_open"].between(
        0,
        30,
        inclusive="both",
    )
)

df_kalshi_timestamped["valid_time_to_expiry"] = (
    df_kalshi_timestamped["tau_raw"] > 0
)

df_kalshi_timestamped["valid_price"] = (
    df_kalshi_timestamped["prob_kalshi"].between(
        0,
        1,
        inclusive="both",
    )
)

df_kalshi_timestamped["timestamped_main_sample"] = (
    df_kalshi_timestamped["trade_within_30m"]
    & df_kalshi_timestamped["valid_time_to_expiry"]
    & df_kalshi_timestamped["valid_price"]
)


# ------------------------------------------------------------
# Define final samples
# ------------------------------------------------------------

df_kalshi_timestamped_15m = (
    df_kalshi_timestamped.loc[
        df_kalshi_timestamped["trade_within_15m"]
        & df_kalshi_timestamped["valid_time_to_expiry"]
        & df_kalshi_timestamped["valid_price"]
    ]
    .copy()
)

df_kalshi_timestamped_30m = (
    df_kalshi_timestamped.loc[
        df_kalshi_timestamped["timestamped_main_sample"]
    ]
    .copy()
)

# Optional compatibility alias.
df_kalshi_timestamped_main = df_kalshi_timestamped_30m.copy()


# ------------------------------------------------------------
# Validate before saving
# ------------------------------------------------------------

assert df_kalshi_timestamped["ticker"].is_unique
assert df_kalshi_timestamped["event_ticker"].notna().all()
assert df_kalshi_timestamped["open_time"].notna().all()
assert df_kalshi_timestamped["close_time"].notna().all()
assert df_kalshi_timestamped["observation_time"].notna().all()
assert df_kalshi_timestamped["valid_time_to_expiry"].all()
assert df_kalshi_timestamped["valid_price"].all()

assert df_kalshi_timestamped_15m["ticker"].is_unique
assert df_kalshi_timestamped_30m["ticker"].is_unique

assert (
    df_kalshi_timestamped_15m["minutes_after_open"] <= 15
).all()

assert (
    df_kalshi_timestamped_30m["minutes_after_open"] <= 30
).all()


# ------------------------------------------------------------
# Save processed and final datasets
# ------------------------------------------------------------

kalshi_timestamped_path = (
    PROCESSED_DIR / "kalshi_kxbtc_first_trades_timestamped.csv"
)

kalshi_timestamped_15m_path = (
    PROCESSED_DIR / "kalshi_kxbtc_first_trades_timestamped_15m.csv"
)

kalshi_timestamped_30m_path = (
    PROCESSED_DIR / "kalshi_kxbtc_first_trades_timestamped_30m.csv"
)

kalshi_timestamped_final_path = (
    FINAL_DIR / "kalshi_kxbtc_first_trades_timestamped_30m.csv"
)

df_kalshi_timestamped.to_csv(
    kalshi_timestamped_path,
    index=False,
)

df_kalshi_timestamped_15m.to_csv(
    kalshi_timestamped_15m_path,
    index=False,
)

df_kalshi_timestamped_30m.to_csv(
    kalshi_timestamped_30m_path,
    index=False,
)

df_kalshi_timestamped_30m.to_csv(
    kalshi_timestamped_final_path,
    index=False,
)


# ------------------------------------------------------------
# Output diagnostics
# ------------------------------------------------------------

print("=" * 70)
print("Timestamped Kalshi first-trade samples")
print("=" * 70)

print("Complete reconciled sample:")
print(f"  Rows: {len(df_kalshi_timestamped):,}")
print(
    f"  Events: "
    f"{df_kalshi_timestamped['event_ticker'].nunique():,}"
)

print("\nFirst trade within 15 minutes:")
print(f"  Rows: {len(df_kalshi_timestamped_15m):,}")
print(
    f"  Events: "
    f"{df_kalshi_timestamped_15m['event_ticker'].nunique():,}"
)

print("\nMain sample: first trade within 30 minutes:")
print(f"  Rows: {len(df_kalshi_timestamped_30m):,}")
print(
    f"  Events: "
    f"{df_kalshi_timestamped_30m['event_ticker'].nunique():,}"
)

print("\nFirst-trade timing:")
display(
    df_kalshi_timestamped[
        ["minutes_after_open", "minutes_to_close"]
    ].describe()
)

print("\nMain-sample prices:")
display(
    df_kalshi_timestamped_30m["prob_kalshi"].describe()
)

print("\nMain sample by strike type:")
print(
    df_kalshi_timestamped_30m[
        "strike_type"
    ].value_counts(dropna=False)
)

print("\nSaved files:")
print(kalshi_timestamped_path)
print(kalshi_timestamped_15m_path)
print(kalshi_timestamped_30m_path)
print(kalshi_timestamped_final_path)

print("\nTimestamped first-trade construction passed.")

Metadata source:
  Rows: 45,257
  Events: 486

Trade-metadata coverage:
  First-trade tickers: 3,802
  Tickers without metadata: 0
Timestamped Kalshi first-trade samples
Complete reconciled sample:
  Rows: 3,802
  Events: 485

First trade within 15 minutes:
  Rows: 2,554
  Events: 483

Main sample: first trade within 30 minutes:
  Rows: 3,135
  Events: 485

First-trade timing:


,minutes_after_open,minutes_to_close
count,3802.000000,3802.000000
mean,13.948284,46.051716
std,15.435304,15.435304
min,0.036169,0.013265
25%,2.489509,38.883329
50%,6.669984,53.330016
75%,21.116671,57.510491
max,59.986735,59.963831



Main-sample prices:


count    3135.000000
mean        0.184715
std         0.169293
min         0.010000
25%         0.050000
50%         0.150000
75%         0.270000
max         0.960000
Name: prob_kalshi, dtype: float64


Main sample by strike type:
strike_type
between    3080
less         40
greater      15
Name: count, dtype: int64

Saved files:
/Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/kalshi_kxbtc_first_trades_timestamped.csv
/Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/kalshi_kxbtc_first_trades_timestamped_15m.csv
/Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/kalshi_kxbtc_first_trades_timestamped_30m.csv
/Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final/kalshi_kxbtc_first_trades_timestamped_30m.csv

Timestamped first-trade construction passed.


In [115]:
# ============================================================
# Final Kalshi timestamped export summary
# ============================================================

kalshi_trades_raw_path = (
    RAW_DIR / "kalshi_kxbtc_timestamped_trades.csv"
)

kalshi_trades_status_path = (
    RAW_DIR / "kalshi_kxbtc_timestamped_trades_status.csv"
)

kalshi_timestamped_path = (
    PROCESSED_DIR / "kalshi_kxbtc_first_trades_timestamped.csv"
)

kalshi_timestamped_15m_path = (
    PROCESSED_DIR / "kalshi_kxbtc_first_trades_timestamped_15m.csv"
)

kalshi_timestamped_30m_path = (
    PROCESSED_DIR / "kalshi_kxbtc_first_trades_timestamped_30m.csv"
)

kalshi_timestamped_final_path = (
    FINAL_DIR / "kalshi_kxbtc_first_trades_timestamped_30m.csv"
)

kalshi_sample_summary_path = (
    TABLES_DIR / "kalshi_timestamped_sample_summary.csv"
)

kalshi_sample_summary = pd.DataFrame([
    {
        "sample": "Complete reconciled first trades",
        "rows": len(df_kalshi_timestamped),
        "events": (
            df_kalshi_timestamped["event_ticker"].nunique()
        ),
        "mean_price": (
            df_kalshi_timestamped["prob_kalshi"].mean()
        ),
    },
    {
        "sample": "First trade within 15 minutes",
        "rows": len(df_kalshi_timestamped_15m),
        "events": (
            df_kalshi_timestamped_15m[
                "event_ticker"
            ].nunique()
        ),
        "mean_price": (
            df_kalshi_timestamped_15m[
                "prob_kalshi"
            ].mean()
        ),
    },
    {
        "sample": "First trade within 30 minutes",
        "rows": len(df_kalshi_timestamped_30m),
        "events": (
            df_kalshi_timestamped_30m[
                "event_ticker"
            ].nunique()
        ),
        "mean_price": (
            df_kalshi_timestamped_30m[
                "prob_kalshi"
            ].mean()
        ),
    },
])

kalshi_sample_summary.to_csv(
    kalshi_sample_summary_path,
    index=False,
)

required_output_paths = [
    kalshi_trades_raw_path,
    kalshi_trades_status_path,
    kalshi_timestamped_path,
    kalshi_timestamped_15m_path,
    kalshi_timestamped_30m_path,
    kalshi_timestamped_final_path,
    kalshi_sample_summary_path,
]

missing_output_paths = [
    path for path in required_output_paths
    if not path.exists()
]

if missing_output_paths:
    raise FileNotFoundError(
        "Missing Kalshi output files:\n"
        + "\n".join(map(str, missing_output_paths))
    )

print("=" * 70)
print("Final Kalshi timestamped export summary")
print("=" * 70)

display(
    kalshi_sample_summary.round(
        {"mean_price": 6}
    )
)

print("\nRaw inputs:")
print("  Historical trades:", kalshi_trades_raw_path)
print("  Download status:  ", kalshi_trades_status_path)

print("\nProcessed outputs:")
print("  Complete sample:", kalshi_timestamped_path)
print("  15-minute sample:", kalshi_timestamped_15m_path)
print("  30-minute sample:", kalshi_timestamped_30m_path)

print("\nFinal output:")
print("  Main 30-minute dataset:", kalshi_timestamped_final_path)

print("\nSummary table:")
print(" ", kalshi_sample_summary_path)

print("\nQuality checks:")
print(
    "  Duplicate main tickers:",
    df_kalshi_timestamped_30m["ticker"].duplicated().sum(),
)
print(
    "  Missing main event tickers:",
    df_kalshi_timestamped_30m["event_ticker"].isna().sum(),
)
print(
    "  Invalid main prices:",
    (
        ~df_kalshi_timestamped_30m["prob_kalshi"]
        .between(0, 1, inclusive="both")
    ).sum(),
)
print(
    "  Main trades after 30 minutes:",
    (
        df_kalshi_timestamped_30m["minutes_after_open"] > 30
    ).sum(),
)

print("\n02_kalshi_data.ipynb completed successfully.")

Final Kalshi timestamped export summary


,sample,rows,events,mean_price
0,Complete reconciled first trades,3802,485,0.173443
1,First trade within 15 minutes,2554,483,0.194988
2,First trade within 30 minutes,3135,485,0.184715



Raw inputs:
  Historical trades: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw/kalshi_kxbtc_timestamped_trades.csv
  Download status:   /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw/kalshi_kxbtc_timestamped_trades_status.csv

Processed outputs:
  Complete sample: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/kalshi_kxbtc_first_trades_timestamped.csv
  15-minute sample: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/kalshi_kxbtc_first_trades_timestamped_15m.csv
  30-minute sample: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/kalshi_kxbtc_first_trades_timestamped_30m.csv

Final output:
  Main 30-minute dataset: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final/kalshi_kxbtc_first_trades_timestamped_30m.csv

Summary table:
  /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/kalshi_timestamped_sample_summary.csv

Quality checks:
  Duplicate 